# Step 3b: Group 10's NRMS-inspired model, corrected reimplementation

This notebook rebuilds the strongest model from the assigned past project ([bt422210/BT4222-Group-10](https://github.com/bt422210/BT4222-Group-10), commit `3007ee3`) and runs it on our split, so it can be compared with our models on identical candidates and metrics.

It is a **corrected reimplementation**, not an exact reproduction: their pipeline stores intermediate files on their own Google Drive, so we rebuild every input from the raw MIND files with their recipes, and we fix the defects listed below.

### What their model is

Despite the name, this is not standard NRMS (which applies self-attention over title words). It is a hybrid ranker:

```
news article ─┬─ text tokens:   [MiniLM title ⊕ abstract (768-d)] + [category-cluster embedding]  → 2-token self-attention → 128-d
              └─ entity tokens: [KG title, KG abstract (100-d)] + [NER title, NER abstract (384-d)] → masked 4-token self-attention → 128-d
              → fusion layer → news vector (128-d)

user          ─ last 10 clicked articles → news encoder → self-attention block → additive attention → user vector (128-d)
                (learnable default vector when the history is empty)

scorer (MLP)  ─ [user vector, news vector, user numeric features, 6 user categorical features,
                 50 TF-IDF topics of the candidate, hour sin/cos, habit score, article recency] → 128 → 64 → 32 → 1
```

Training: one click plus four non-clicked candidates sampled from the same impression, softmax cross-entropy over the five, Adam (lr 5×10⁻⁴), batch 32, 10 epochs. We use their **Test_2 configuration (2 heads, dropout 0.1)**, the one with the best validation score among their four runs.

### Kept exactly as Group 10 built it

| Component | Recipe |
|---|---|
| Split | Same as theirs and ours: train Nov 9–12, validation Nov 13, test Nov 14 |
| Text vectors | `all-MiniLM-L6-v2` on title and on abstract (title when missing), each normalised, concatenated, renormalised |
| Entity vectors | Mean KG embedding of title / abstract WikidataIds; spaCy `en_core_web_sm` entity phrases encoded with MiniLM; four presence flags |
| Categories | Subcategories with ≥ 75th-percentile size keep their label, else `<category>_general`; labels under 100 articles → `miscellaneous`; 8 largest split into 3 KMeans clusters |
| Topics | TF-IDF (1–2-grams, 1,000 terms) on titles → 50 SVD components |
| User features | History length (log, scaled), category diversity and concentration, top-3 and recent favourite category, diversity cluster (KMeans 3), interest cluster (MiniBatchKMeans 4), all fitted on training rows |
| Negatives | 4 per click, from the same impression, with replacement when fewer than 4 exist |
| Architecture and hyperparameters | Layer for layer, see `shared/nrms_g10.py` |

Category merging, clustering and topics are fitted on the whole article catalogue, as Group 10 did. They use article text only, never labels or clicks.

### Corrected

| # | Issue in Group 10's code | Evidence | Correction |
|---|---|---|---|
| 1 | Training recency scores do not line up with the candidates | `HybridNewsDataset.__getitem__` re-shuffles candidates with a new permutation but takes `recency_scores[idx]` in the context table's own order | Recency is computed per candidate row and travels with it |
| 2 | Context features are fetched by row position from tables that were re-sorted | `3_feature_engineering` sorts the context tables by user and time before saving; user features keep the original order; the model indexes both by position. Their parquet conversion is not in the repo, so this is *probable*, not confirmed | Every input is joined by `impression_id` or news id |
| 3 | Recency ignores exposures outside the sampled training rows | First-seen time comes from the 1+4 sampled training candidates only; any article first shown on Nov 13 or 14 keeps recency 1.0 however old it is | First exposure anywhere in the log strictly before the impression |
| 4 | Habit score resets at each split boundary | Previous visit is looked up within the same split | Previous impression anywhere in the log strictly before t |
| 5 | Test uses the last epoch | A best epoch is recorded but its weights are not restored | Best validation epoch is restored before scoring test |
| 6 | No random seeds | None set anywhere | Seeds fixed |
| 7 | Evaluation not comparable | Pooled AUC, MRR from the first click only, and stable sorting that favours clicks on ties | MIND-standard per-impression metrics with a seeded tie-break; their legacy metrics are also shown for comparison |
| 8 | Headline numbers were training scores on 1+4 sampled slates | Their summary table | Train, validation and test are all scored on full slates |

> Metrics here come from `shared/eval_interim.py`, a stand-in for tuning until the team evaluator exists. The numbers in the final report come from `step6_evaluation_diagnostics.ipynb`.

## 1. Setup

In [1]:
import os, sys, subprocess, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.getcwd()
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from shared import schema as S
from shared.predictions import write_predictions
from shared.nrms_inputs import build_inputs
from shared import nrms_g10 as N
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = "REAL"   # recorded in every prediction file
import kagglehub
RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
TEXT_METHOD, NER_METHOD, CAT_KWARGS = "minilm", "spacy", {}
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
CACHE_DIR = os.path.join(DATA_ROOT, "nrms_cache", DATA_KIND.lower())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 4222

CONFIG = dict(num_heads=2, dropout=0.1, user_heads=2, user_dropout=0.1)   # Group 10 Test_2
TRAIN = dict(epochs=10, lr=5e-4, batch_size=32)
SELECT = "ndcg@5"   # Group 10's own checkpoint criterion
MODEL_ID = "nrms_g10"
print(f"DATA_KIND={DATA_KIND} | device={DEVICE} | raw={RAW_DIR}")

C:\Users\Bryan Ng\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_KIND=REAL | device=cpu | raw=C:\Users\Bryan Ng\.cache\kagglehub\datasets\arashnic\mind-news-dataset\versions\2\MINDsmall_train


## 2. Candidate rows and labels from the shared metadata tables

Every prediction must carry the shared `row_id`, so the candidate lists come from `meta_{split}.parquet` rather than from re-parsing `behaviors.tsv`.

In [2]:
META_COLS = ["row_id", "impression_id", "news_id", "time", "label", "is_cold_10"]
meta = {s: pd.read_parquet(S.meta_path(FEATURE_ROOT, s), columns=META_COLS) for s in S.SPLITS}
GROUP10_IMPRESSIONS = {"train": 95071, "val": 31624, "test": 30270}
pd.DataFrame({s: {"impressions": m["impression_id"].nunique(), "Group 10 impressions": GROUP10_IMPRESSIONS[s],
                  "candidate rows": len(m), "CTR": round(m["label"].mean(), 4)} for s, m in meta.items()}).T

,impressions,Group 10 impressions,candidate rows,CTR
train,95071.0,95071.0,3389342.0,0.0418
val,31624.0,31624.0,1231673.0,0.0389
test,30270.0,30270.0,1222429.0,0.0383


## 3. Build Group 10's inputs

The MiniLM and spaCy steps are the slow part (roughly 10–30 minutes on free Colab) and are cached on Drive, so later runs skip them. The printed counts can be checked against Group 10's notebooks: 79 merged categories, 95 sub-clustered labels, and 141,558 training samples.

In [3]:
t0 = time.time()
inputs = build_inputs(RAW_DIR, meta, text_method=TEXT_METHOD, ner_method=NER_METHOD, device=DEVICE,
                      cache_dir=CACHE_DIR, cat_kwargs=CAT_KWARGS, seed=SEED)
print(f"built in {time.time() - t0:.0f}s")
for split in S.SPLITS:   # every candidate row is attached to its own impression
    assert (inputs["impressions"]["impression_id"][inputs["splits"][split]["imp"]] == meta[split]["impression_id"].to_numpy()).all()
flags = inputs["news"]["flags"].mean(0)
pd.Series({**inputs["info"], "user vocab sizes": inputs["user_vocab_sizes"],
           "share with KG title / KG abstract / NER title / NER abstract": np.round(flags, 3).tolist()})

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


built in 947s


subcat_threshold                                                                                              110
n_categories                                                                                                   79
n_subclusters                                                                                                  95
n_train_samples                                                                                            141558
text_method                                                                                                minilm
ner_method                                                                                                  spacy
user vocab sizes                                                {'recent_fav_cat': 96, 'top_cat_1': 96, 'top_c...
share with KG title / KG abstract / NER title / NER abstract    [0.7239999771118164, 0.7269999980926514, 0.885...
dtype: object

**Check.** The rebuilt inputs match Group 10's notebooks: 79 merged categories, 95 sub-clustered labels and 141,558 training samples (one click plus four negatives each).

## 4. Train with Group 10's loop, selecting the epoch on validation

In [4]:
N.set_seed(SEED)
model = N.NRMSG10(inputs, **CONFIG).to(DEVICE)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10"]
val_meta, val_split = meta["val"], inputs["splits"]["val"]
def val_fn(m):
    res = E.evaluate(val_meta, N.score_split(m, val_split))
    return {k: res[k] for k in SHOW}

history, best_epoch = N.train(model, inputs["train_samples"], seed=SEED, val_fn=val_fn, select_metric=SELECT, **TRAIN)
print(f"best validation {SELECT} at epoch {best_epoch}; those weights are now loaded")
history.round(4)

trainable parameters: 863,329


epoch=1, train_loss=1.4566, train_seconds=126.8000, val_auc=0.5616, val_mrr=0.2485, val_ndcg@5=0.2591, val_ndcg@10=0.3203, val_cold_recall@10=0.5949


epoch=2, train_loss=1.4025, train_seconds=127.9000, val_auc=0.5955, val_mrr=0.2712, val_ndcg@5=0.2888, val_ndcg@10=0.3485, val_cold_recall@10=0.6549


epoch=3, train_loss=1.3719, train_seconds=127.7000, val_auc=0.6056, val_mrr=0.2806, val_ndcg@5=0.2993, val_ndcg@10=0.3596, val_cold_recall@10=0.6281


epoch=4, train_loss=1.3542, train_seconds=128.5000, val_auc=0.6106, val_mrr=0.2822, val_ndcg@5=0.3015, val_ndcg@10=0.3604, val_cold_recall@10=0.5728


epoch=5, train_loss=1.3413, train_seconds=129.7000, val_auc=0.6184, val_mrr=0.2875, val_ndcg@5=0.3076, val_ndcg@10=0.3679, val_cold_recall@10=0.5748


... (2 outputs left out)

epoch=8, train_loss=1.3122, train_seconds=124.8000, val_auc=0.6272, val_mrr=0.2929, val_ndcg@5=0.3155, val_ndcg@10=0.3740, val_cold_recall@10=0.5886


epoch=9, train_loss=1.3024, train_seconds=125.8000, val_auc=0.6279, val_mrr=0.2928, val_ndcg@5=0.3148, val_ndcg@10=0.3745, val_cold_recall@10=0.5604


epoch=10, train_loss=1.2946, train_seconds=125.1000, val_auc=0.6365, val_mrr=0.3018, val_ndcg@5=0.3254, val_ndcg@10=0.3829, val_cold_recall@10=0.5803
best validation ndcg@5 at epoch 10; those weights are now loaded


,train_loss,train_seconds,val_auc,val_mrr,val_ndcg@5,val_ndcg@10,val_cold_recall@10
epoch,,,,,,,
1,1.4566,126.8,0.5616,0.2485,0.2591,0.3203,0.5949
2,1.4025,127.9,0.5955,0.2712,0.2888,0.3485,0.6549
3,1.3719,127.7,0.6056,0.2806,0.2993,0.3596,0.6281
4,1.3542,128.5,0.6106,0.2822,0.3015,0.3604,0.5728
5,1.3413,129.7,0.6184,0.2875,0.3076,0.3679,0.5748
6,1.3319,127.5,0.6168,0.2886,0.3082,0.3686,0.5847
7,1.3212,128.9,0.6309,0.2956,0.3180,0.3762,0.5432
8,1.3122,124.8,0.6272,0.2929,0.3155,0.3740,0.5886
9,1.3024,125.8,0.6279,0.2928,0.3148,0.3745,0.5604


**Takeaway: training.** Validation nDCG@10 rises in most epochs (0.320 → 0.383) and is highest at the last one, so epoch 10 is selected and the model had probably not converged. Ten epochs is Group 10's setting, kept for fidelity; training longer might help, but it would no longer be their model. Validation Cold Recall@10 follows a different path: it peaks at epoch 2 (0.65) and moves between 0.54 and 0.63 afterwards.

## 5. Score every split with the selected epoch

Train, validation and test are scored on their full slates and written as prediction files. Group 10's own legacy metrics (pooled AUC, first-click MRR) are computed alongside, so this run can be lined up against their saved test results for the same configuration.

In [5]:
def legacy_metrics(m, score):
    """Group 10's definitions: AUC pooled over all rows; MRR of the first click per impression."""
    d = m[["impression_id", "label", "row_id"]].assign(s=score, k=E.tie_key(m["row_id"].to_numpy()))
    d = d.sort_values(["impression_id", "s", "k"], ascending=[True, False, True])
    d["rank"] = d.groupby("impression_id").cumcount() + 1
    first = d[d["label"] == 1].groupby("impression_id")["rank"].min()
    return {"pooled_auc": roc_auc_score(m["label"], score), "first_click_mrr": float((1 / first).mean())}

config_id = f"heads={CONFIG['num_heads']},dropout={CONFIG['dropout']},epoch={best_epoch}"
rows = []
for split in S.SPLITS:
    sc = N.score_split(model, inputs["splits"][split])
    write_predictions(meta[split]["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=MODEL_ID,
                      config_id=config_id, split=split, schema_version=S.SCHEMA_VERSION,
                      extra={"selected_on": "val", "selection_metric": SELECT, "group10_config": "Test_2"})
    res = E.evaluate(meta[split], sc)
    rows.append({"split": split, **{k: res[k] for k in SHOW + ["n_cold_click_impressions"]}, **legacy_metrics(meta[split], sc)})
results = pd.DataFrame(rows).set_index("split")
results.round(4)

,auc,mrr,ndcg@5,ndcg@10,cold_recall@10,n_cold_click_impressions,pooled_auc,first_click_mrr
split,,,,,,,,
train,0.7461,0.4061,0.4449,0.4988,0.6344,3223,0.5759,0.4635
val,0.6365,0.3018,0.3254,0.3829,0.5803,938,0.5405,0.3456
test,0.6094,0.2658,0.2815,0.3439,0.5743,837,0.5339,0.3092


In [6]:
group10_test = pd.Series({"pooled_auc": 0.7046, "first_click_mrr": 0.2951, "ndcg@5": 0.2699, "ndcg@10": 0.3319},
                         name="Group 10 saved test (Test_2, last epoch)")
ours = results.loc["test", group10_test.index].rename("this reimplementation, test")
pd.concat([group10_test, ours], axis=1).round(4)

,"Group 10 saved test (Test_2, last epoch)","this reimplementation, test"
pooled_auc,0.7046,0.5339
first_click_mrr,0.2951,0.3092
ndcg@5,0.2699,0.2815
ndcg@10,0.3319,0.3439


**Takeaway: fidelity.** On the ranking metrics the reimplementation lands close to Group 10's saved test results, slightly above them: nDCG@10 0.344 vs 0.332, nDCG@5 0.282 vs 0.270, first-click MRR 0.309 vs 0.295. That fits the corrections (best-epoch restore, aligned recency) helping a little. Pooled AUC is much lower (0.53 vs 0.70). Pooled AUC compares scores across impressions, which a model trained with a within-impression softmax is not built to do; its per-impression AUC is 0.61. We could not establish why their pooled AUC was higher (correction 3, the stale recency of articles first shown on Nov 13 or 14, is one candidate), so the gap is reported rather than explained away.

## 6. What the reimplementation shows

* **Overall it ranks worse than the smoothed-CTR baselines** (test nDCG@10 0.344 vs 0.357, AUC 0.61 vs 0.65). Without click counts or CTR, it cannot tell which articles are doing well right now.
* **It is far better on cold articles:** test Cold Recall@10 0.574, almost double the best baseline (0.298, category CTR). Its scores come from content and the user's history, so a zero-exposure article is judged on what it is about rather than on missing behaviour.
* **There is a clear train–validation gap** (AUC 0.746 vs 0.637, both on full slates), so the model fits the training days noticeably better than new ones.
* All later comparisons use the MIND-standard columns, not Group 10's legacy metrics.
* This sets up step 4: the baselines have behaviour but no content, and this model has content but no behaviour. Does combining them beat both?

*Based on the run with feature schema v0.1 and the interim evaluator. The final numbers and intervals come from step 6.*